<a href="https://colab.research.google.com/github/ayyucedemirbas/Spatial_Transcriptomics_Analysis/blob/main/SquidPy_ST_analysis_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install squidpy

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.8/87.8 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of spatialdata to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of spatialdata to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 kB 2.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of ome-zarr to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is comp

In [2]:
!pip install scanpy anndata

In [3]:
!pip install omnipath

In [4]:
!pip install leidenalg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 53.1 MB/s eta 0:00:00


In [5]:
import squidpy as sq
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as SklearnPCA
from pathlib import Path
import scipy.sparse as sp
import sys

import warnings
warnings.filterwarnings("ignore")

/usr/local/lib/python3.12/dist-packages/docrep/decorators.py:43: SyntaxWarning: 'n_jobs' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/usr/local/lib/python3.12/dist-packages/docrep/decorators.py:43: SyntaxWarning: 'show_progress_bar' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


In [6]:
output_dir = Path("squidpy_output")
output_dir.mkdir(parents=True, exist_ok=True)

In [7]:
sc.settings.figdir= str(output_dir)

In [8]:
adata = sq.datasets.visium_hne_adata()
img = sq.datasets.visium_hne_image()

INFO     Downloading visium_hne_adata.h5ad from https://exampledata.scverse.org/squidpy/visium_hne_adata.h5ad      


  0%|                                               | 0.00/329M [00:00<?, ?B/s]

INFO     Downloading visium_hne_image.tiff from https://exampledata.scverse.org/squidpy/visium_hne_image.tiff      


  0%|                                               | 0.00/398M [00:00<?, ?B/s]

In [9]:
adata

AnnData object with n_obs × n_vars = 2688 × 18078
    obs: 'in_tissue', 'array_row', 'array_col', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_counts', 'leiden', 'cluster'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'cluster_colors', 'hvg', 'leiden', 'leiden_colors', 'neighbors', 'pca', 'rank_genes_groups', 'spatial', 'umap'
    obsm: 'X_pca', 'X_umap', 'spatial'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [10]:
list(adata.obsm.keys())

['X_pca', 'X_umap', 'spatial']

In [11]:
list(adata.uns.keys())

['cluster_colors',
 'hvg',
 'leiden',
 'leiden_colors',
 'neighbors',
 'pca',
 'rank_genes_groups',
 'spatial',
 'umap']

In [12]:
adata.obsm['spatial'].shape

(2688, 2)

In [13]:
adata.obs.head()

,in_tissue,array_row,array_col,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,total_counts_mt,log1p_total_counts_mt,pct_counts_mt,n_counts,leiden,cluster
AAACAAGTATCTCCCA-1,1,50,102,4928,8.502891,19340.0,9.869983,38.428128,43.133402,49.214064,60.449845,3857.0,8.257904,19.943123,19340.0,2,Cortex_2
AAACAATCTACTAGCA-1,1,3,43,3448,8.145840,13750.0,9.528867,50.516364,55.141818,60.952727,70.574545,3267.0,8.091933,23.760000,13750.0,11,Cortex_5
AAACACCAATAACTGC-1,1,59,19,6022,8.703341,32710.0,10.395467,40.483033,47.071232,54.564353,65.087129,4910.0,8.499233,15.010699,32710.0,7,Thalamus_2
AAACAGAGCGACTCCT-1,1,14,94,4311,8.369157,15909.0,9.674704,40.957948,45.810547,52.077440,62.976931,3270.0,8.092851,20.554403,15909.0,11,Cortex_5
AAACCGGGTAGGTACC-1,1,42,28,5787,8.663542,31856.0,10.369013,40.287544,45.887745,52.982170,64.248493,6693.0,8.808967,21.010170,31856.0,7,Thalamus_2


In [14]:
adata.var.head()

,gene_ids,feature_types,genome,mt,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts,n_cells,highly_variable,highly_variable_rank,means,variances,variances_norm
Xkr4,ENSMUSG00000051951,Gene Expression,mm10,False,233,0.093032,0.088955,91.363973,251.0,5.529429,233,False,NaN,0.093378,0.098832,0.815019
Sox17,ENSMUSG00000025902,Gene Expression,mm10,False,298,0.128243,0.120662,88.954781,346.0,5.849325,298,False,NaN,0.128720,0.156108,0.931378
Mrpl15,ENSMUSG00000033845,Gene Expression,mm10,False,1775,1.370645,0.863162,34.210526,3698.0,8.215817,1775,False,NaN,1.375744,2.163193,0.850736
Lypla1,ENSMUSG00000025903,Gene Expression,mm10,False,1294,0.741290,0.554626,52.038547,2000.0,7.601402,1293,False,NaN,0.743676,0.984143,0.873699
Tcea1,ENSMUSG00000033813,Gene Expression,mm10,False,1975,1.727947,1.003549,26.797628,4662.0,8.447414,1974,False,NaN,1.734003,3.030447,0.856292


In [15]:
img

ImageContainer[shape=(11757, 11291), layers=['image']]

In [16]:
adata.var["mt"] = adata.var_names.str.startswith("mt-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

In [17]:
print(f"Total counts per spot: {adata.obs['total_counts'].describe().to_string()}")
print(f"Genes per spot: {adata.obs['n_genes_by_counts'].describe().to_string()}")
print(f"% mitochondrial:{adata.obs['pct_counts_mt'].describe().to_string()}")

Total counts per spot: count    2688.000000
mean     6690.863281
std       851.619934
min      3031.893311
25%      6186.069092
50%      6838.481689
75%      7308.461548
max      8852.813477
Genes per spot: count     2688.000000
mean      5733.603795
std       1459.501141
min       1461.000000
25%       4749.500000
50%       5805.000000
75%       6754.000000
max      10683.000000
% mitochondrial:count    2688.000000
mean        0.969446
std         0.184712
min         0.553792
25%         0.848501
50%         0.923233
75%         1.038620
max         2.209685


In [18]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

In [19]:
axes[0,0].hist(adata.obs["total_counts"], bins=70, color="#4C72B0", alpha=0.85, edgecolor="white")

(array([  1.,   0.,   0.,   1.,   0.,   1.,   0.,   1.,   3.,   1.,   2.,
          1.,   1.,   6.,   4.,   4.,   7.,   6.,  12.,  13.,  18.,   7.,
         13.,  24.,  18.,  14.,  33.,  21.,  35.,  35.,  36.,  42.,  45.,
         51.,  39.,  60.,  61.,  61.,  71.,  84.,  78.,  81.,  97.,  86.,
        100.,  92., 112., 119., 119., 117., 125., 117., 112., 103., 105.,
         81.,  50.,  45.,  35.,  15.,  16.,   9.,  10.,   9.,   7.,   6.,
          3.,   4.,   2.,   1.]),
 array([3031.89331055, 3115.04931641, 3198.20532227, 3281.36132812,
        3364.51733398, 3447.67333984, 3530.8293457 , 3613.98535156,
        3697.14135742, 3780.29736328, 3863.45336914, 3946.609375  ,
        4029.76513672, 4112.92138672, 4196.07714844, 4279.23339844,
        4362.38916016, 4445.54541016, 4528.70117188, 4611.85742188,
        4695.01318359, 4778.16943359, 4861.32519531, 4944.48144531,
        5027.63720703, 5110.79296875, 5193.94921875, 5277.10546875,
        5360.26123047, 5443.41699219, 5526.573

In [20]:
axes[0,0].axvline(adata.obs["total_counts"].median(), color="#E74C3C", linestyle="--", lw=2,
                   label=f"Median: {adata.obs['total_counts'].median():,.0f}")
axes[0,0].set_xlabel("Total UMI counts", fontsize=11)
axes[0,0].set_ylabel("Number of spots",   fontsize=11)
axes[0,0].set_title("Total Counts per Spot", fontsize=12, fontweight="bold")
axes[0,0].legend(fontsize=9)

axes[0,1].hist(adata.obs["n_genes_by_counts"], bins=70, color="#DD8452", alpha=0.85, edgecolor="white")
axes[0,1].axvline(adata.obs["n_genes_by_counts"].median(), color="#E74C3C", linestyle="--", lw=2,
                   label=f"Median: {adata.obs['n_genes_by_counts'].median():,.0f}")
axes[0,1].set_xlabel("Genes detected",   fontsize=11)
axes[0,1].set_ylabel("Number of spots",  fontsize=11)
axes[0,1].set_title("Genes per Spot",    fontsize=12, fontweight="bold")
axes[0,1].legend(fontsize=9)

axes[0,2].scatter(adata.obs["total_counts"], adata.obs["n_genes_by_counts"],
                   alpha=0.35, s=9, c="#55A868")
axes[0,2].set_xlabel("Total UMI counts",  fontsize=11)
axes[0,2].set_ylabel("Genes detected",    fontsize=11)
axes[0,2].set_title("Counts vs Genes",    fontsize=12, fontweight="bold")

tot   = np.array(adata.X.sum(axis=0)).flatten()
tidx  = np.argsort(tot)[::-1][:20]
tpct  = tot[tidx] / tot.sum() * 100
bar_c = ["#E74C3C" if adata.var_names[i].startswith("mt-") else "#8172B3" for i in tidx]
axes[1,0].barh(range(20), tpct[::-1], color=bar_c[::-1], alpha=0.85)
axes[1,0].set_yticks(range(20))
axes[1,0].set_yticklabels(adata.var_names[tidx][::-1], fontsize=7)
axes[1,0].set_xlabel("% total UMI counts", fontsize=11)
axes[1,0].set_title("Top 20 Expressed Genes", fontsize=12, fontweight="bold")

axes[1,1].hist(adata.obs["pct_counts_mt"], bins=60, color="#C44E52", alpha=0.85, edgecolor="white")
axes[1,1].axvline(adata.obs["pct_counts_mt"].median(), color="#2C3E50", linestyle="--", lw=2,
                   label=f"Median: {adata.obs['pct_counts_mt'].median():.2f}%")
axes[1,1].set_xlabel("% mitochondrial counts", fontsize=11)
axes[1,1].set_ylabel("Number of spots",         fontsize=11)
axes[1,1].set_title("Mitochondrial %",           fontsize=12, fontweight="bold")
axes[1,1].legend(fontsize=9)

cells_per_gene = np.array((adata.X > 0).sum(axis=0)).flatten()
axes[1,2].hist(cells_per_gene, bins=80, color="#2ECC71", alpha=0.85, edgecolor="white", log=True)
axes[1,2].set_xlabel("Spots expressing gene", fontsize=11)
axes[1,2].set_ylabel("Gene count (log scale)", fontsize=11)
axes[1,2].set_title("Gene Detection Frequency", fontsize=12, fontweight="bold")

plt.suptitle("Quality Control — 10x Visium Mouse Brain (H&E)", fontsize=14,
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(output_dir / "01_qc_metrics.png", dpi=150, bbox_inches="tight")
plt.close()

In [21]:
#filtering
sc.pp.filter_cells(adata, min_counts=5_000)
sc.pp.filter_cells(adata, max_counts=35_000)
sc.pp.filter_cells(adata, min_genes=300)
sc.pp.filter_genes(adata,  min_cells=10)

In [22]:
adata.n_obs

2572

In [23]:
adata.n_vars

16939

In [24]:
adata.layers["counts"]  = adata.X.copy()

sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X.copy()

sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=2000)

In [25]:
adata.var['highly_variable'].sum()

np.int64(2000)

In [26]:
fig, ax = plt.subplots(figsize=(10, 5))
sc.pl.highly_variable_genes(adata, show=False)
ax.set_title("Highly Variable Genes — Seurat flavour, top 2000", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(output_dir / "02_hvg_selection.png", dpi=150, bbox_inches="tight")
plt.close()

In [27]:
adata.raw = adata

sc.pp.scale(adata, zero_center=False, max_value=10)
sc.pp.pca(adata, n_comps=50, use_highly_variable=True, svd_solver="arpack", random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cumvar = np.cumsum(adata.uns["pca"]["variance_ratio"]) * 100
axes[0].plot(cumvar, "o-", ms=4, color="#4C72B0", lw=1.5)
axes[0].axhline(90, color="#E74C3C", linestyle="--", alpha=0.7, label="90% variance")
pc90 = int(np.searchsorted(cumvar, 90)) + 1
axes[0].axvline(pc90, color="#2ECC71", linestyle="--", alpha=0.7, label=f"PC {pc90}")
axes[0].set_xlabel("Principal component",            fontsize=11)
axes[0].set_ylabel("Cumulative variance (%)",        fontsize=11)
axes[0].set_title("PCA Cumulative Variance",         fontsize=12, fontweight="bold")
axes[0].legend(fontsize=9)

axes[1].bar(range(1, 31), adata.uns["pca"]["variance_ratio"][:30] * 100,
            color="#DD8452", alpha=0.85, edgecolor="white")
axes[1].set_xlabel("Principal component",  fontsize=11)
axes[1].set_ylabel("Variance explained (%)", fontsize=11)
axes[1].set_title("Scree Plot (Top 30 PCs)", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig(output_dir / "03_pca_variance.png", dpi=150, bbox_inches="tight")
plt.close()

In [28]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, metric="cosine", random_state=42)
sc.tl.umap(adata,    min_dist=0.3, spread=1.0,    random_state=42)
sc.tl.leiden(adata,  resolution=0.5, key_added="leiden", random_state=42)

adata.X = adata.layers["lognorm"].copy()

n_clusters = adata.obs["leiden"].nunique()
print(f"Leiden clusters (resolution=0.5): {n_clusters}")
print(adata.obs["leiden"].value_counts().sort_index().to_string())

fig, axes = plt.subplots(1, 3, figsize=(24, 7))
sc.pl.umap(adata, color="leiden", ax=axes[0], show=False,
           title="UMAP — Leiden Clusters", legend_loc="on data",
           legend_fontsize=9, legend_fontoutline=2)
sc.pl.umap(adata, color="total_counts", ax=axes[1], show=False,
           title="UMAP - Total Counts", color_map="viridis")
sc.pl.umap(adata, color="pct_counts_mt", ax=axes[2], show=False,
           title="UMAP — % Mitochondrial", color_map="Reds")
plt.suptitle("UMAP Embedding — 10x Visium Mouse Brain", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(output_dir / "04_umap_clustering.png", dpi=150, bbox_inches="tight")
plt.close()


  Leiden clusters (resolution=0.5): 11
leiden
0     494
1     337
2     296
3     264
4     260
5     237
6     218
7     195
8     156
9      83
10     32


In [29]:
sc.tl.rank_genes_groups(
    adata, groupby="leiden", method="wilcoxon",
    use_raw=True, tie_correct=True, key_added="rank_genes_leiden"
)

print("Top marker genes per cluster:")
for cl in sorted(adata.obs["leiden"].unique(), key=int):
    df = sc.get.rank_genes_groups_df(adata, group=cl, key="rank_genes_leiden").head(5)
    genes_str = ", ".join(df["names"].tolist())
    print(f"    Cluster {cl:>2s}:  {genes_str}")

try:
    fig = sc.pl.rank_genes_groups_dotplot(
        adata, n_genes=5, groupby="leiden", key="rank_genes_leiden",
        show=False, return_fig=True, standard_scale="var",
        colorbar_title="Mean z-score"
    )
    plt.suptitle("Top 5 Marker Genes per Leiden Cluster (Wilcoxon)",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.savefig(output_dir / "05_marker_dotplot.png", dpi=150, bbox_inches="tight")
    plt.close()
except Exception:
    plt.close()


  Top marker genes per cluster:
    Cluster  0:  Mef2c, Lamp5, Stx1a, Nrgn, Cabp1
    Cluster  1:  Dlk1, Arhgap36, Gal, Nap1l5, Resp18
    Cluster  2:  Nptxr, Slc30a3, Lmo3, Syn2, Hpcal1
    Cluster  3:  Fa2h, Ugt8a, Ermn, Pdlim2, Mog
    Cluster  4:  Spink8, Cabp7, Klk8, Lrrc10b, Hpca
    Cluster  5:  Shox2, Tnnt1, Synpo2, Plekhg1, Rab37
    Cluster  6:  3110035E14Rik, Trbc2, Hs3st2, Tbr1, Nxph3
    Cluster  7:  Ctxn3, Cbln4, Cbln1, Tcf7l2, Lhx9
    Cluster  8:  Penk, Tac1, Trhr, Sv2c, 6430628N08Rik
    Cluster  9:  Slc22a6, Col1a1, Slc6a13, Fmod, Myh11
    Cluster 10:  Gm32468, Frmpd1os, Defb11, Steap1, Slc16a8


In [30]:
sq.gr.spatial_neighbors(
    adata,
    n_rings=2,
    coord_type="grid",
    n_neighs=6,
    key_added="spatial",
)

conn  = adata.obsp["spatial_connectivities"]
dists = adata.obsp["spatial_distances"]
deg   = np.array(conn.sum(axis=1)).flatten()

INFO     Creating graph using `None` transform and `1` libraries.                                                  


In [31]:
conn.shape[0]

2572

In [32]:
conn.nnz // 2

21270

In [33]:
print(f"Degree — mean: {deg.mean():.2f}  min: {deg.min():.0f}  max: {deg.max():.0f}")

Degree — mean: 16.54  min: 0  max: 18


In [34]:
print(f"Distance range (non-zero): " f"{dists.data.min():.2f} – {dists.data.max():.2f} µm")

Distance range (non-zero): 1.00 – 2.00 µm


In [35]:
adata.uns.pop('leiden_colors', None)

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
sq.pl.spatial_scatter(adata, color="leiden", ax=axes[0], size=1.8,
                       title="Leiden Clusters — Spatial", legend_loc="right margin")
sq.pl.spatial_scatter(adata, color="total_counts", ax=axes[1], size=1.8,
                       title="Total UMI Counts — Spatial")
sq.pl.spatial_scatter(adata, color="n_genes_by_counts", ax=axes[2], size=1.8,
                       title="Genes Detected — Spatial")

plt.suptitle("10x Visium Mouse Brain — Spatial Overview", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(output_dir / "06_spatial_overview.png", dpi=150, bbox_inches="tight")
plt.close()

In [36]:
sq.gr.spatial_autocorr(adata, mode="moran", n_perms=100, n_jobs=1)
moran_df = adata.uns["moranI"].sort_values("I", ascending=False).copy()

sq.gr.spatial_autocorr(adata, mode="geary", n_perms=100, n_jobs=1)
geary_df = adata.uns["gearyC"].sort_values("C").copy()

n_sig_moran = (moran_df["pval_norm"] < 0.05).sum()
n_sig_geary = (geary_df["pval_norm"] < 0.05).sum()

print(f"Moran's I  — genes tested: {len(moran_df):,}  "
      f"significant (p<0.05): {n_sig_moran:,}")
print(f"  Geary's C  — genes tested: {len(geary_df):,}  "
      f"significant (p<0.05): {n_sig_geary:,}")

print(f"Top 15 SVGs (Moran's I):")
print(moran_df[["I", "pval_norm", "pval_z_sim"]].head(15).to_string())

print(f"Top 15 SVGs (Geary's C):")
print(geary_df[["C", "pval_norm", "pval_z_sim"]].head(15).to_string())

moran_df.to_csv(output_dir / "moranI_results.csv")
geary_df.to_csv(output_dir / "gearyC_results.csv")

sig_mask = moran_df["pval_norm"] < 0.05
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(
    moran_df.loc[~sig_mask, "I"],
    -np.log10(moran_df.loc[~sig_mask, "pval_norm"] + 1e-300),
    alpha=0.25, s=10, c="gray", label=f"Not sig. (n={(~sig_mask).sum():,})")
axes[0].scatter(
    moran_df.loc[sig_mask, "I"],
    -np.log10(moran_df.loc[sig_mask, "pval_norm"] + 1e-300),
    alpha=0.7, s=14, c="#E74C3C", label=f"p < 0.05 (n={sig_mask.sum():,})")
for gene in moran_df.head(8).index:
    axes[0].annotate(
        gene,
        (moran_df.loc[gene, "I"], -np.log10(moran_df.loc[gene, "pval_norm"] + 1e-300)),
        fontsize=7, ha="left", va="bottom",
        arrowprops=dict(arrowstyle="-", lw=0.5, color="gray"),
        xytext=(moran_df.loc[gene, "I"] + 0.01,
                -np.log10(moran_df.loc[gene, "pval_norm"] + 1e-300) + 0.5),
    )
axes[0].axhline(-np.log10(0.05), c="steelblue", ls="--", alpha=0.5, label="p = 0.05")
axes[0].set_xlabel("Moran's I statistic",    fontsize=11)
axes[0].set_ylabel("-log₁₀(p-value)",         fontsize=11)
axes[0].set_title("Moran's I Significance",   fontsize=12, fontweight="bold")
axes[0].legend(fontsize=9)

top20 = moran_df.head(20)
colors20 = ["#E74C3C" if v > 0.4 else "#3498DB" for v in top20["I"]]
axes[1].barh(range(20), top20["I"].values[::-1], color=colors20[::-1], alpha=0.85, edgecolor="white")
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(top20.index[::-1], fontsize=8)
axes[1].set_xlabel("Moran's I", fontsize=11)
axes[1].set_title("Top 20 Spatially Variable Genes", fontsize=12, fontweight="bold")

plt.suptitle("Spatial Autocorrelation Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(output_dir / "07_spatial_autocorr_summary.png", dpi=150, bbox_inches="tight")
plt.close()

  0%|          | 0/100 [00:00<?, ?/s]

  0%|          | 0/100 [00:00<?, ?/s]

Moran's I  — genes tested: 2,000  significant (p<0.05): 1,672
  Geary's C  — genes tested: 2,000  significant (p<0.05): 1,855
Top 15 SVGs (Moran's I):
                I  pval_norm  pval_z_sim
Slc17a7  0.787534        0.0         0.0
Nrgn     0.758347        0.0         0.0
Mbp      0.735942        0.0         0.0
Cck      0.734623        0.0         0.0
Itpka    0.715802        0.0         0.0
Agt      0.699068        0.0         0.0
Tcf7l2   0.697136        0.0         0.0
Baiap3   0.688544        0.0         0.0
Prkcd    0.687791        0.0         0.0
Mobp     0.679005        0.0         0.0
Hcrt     0.677128        0.0         0.0
Ddn      0.675502        0.0         0.0
Pmch     0.667962        0.0         0.0
Camk2n1  0.665218        0.0         0.0
Gal      0.652811        0.0         0.0
Top 15 SVGs (Geary's C):
                C  pval_norm     pval_z_sim
Slc17a7  0.211996        0.0   0.000000e+00
Nrgn     0.242194        0.0   0.000000e+00
Cck      0.265044        0.0   0.000

In [37]:
top_svgs = [g for g in moran_df.head(9).index if g in adata.var_names]
n_rows   = (len(top_svgs) + 2) // 3
fig, axes = plt.subplots(n_rows, 3, figsize=(18, 6 * n_rows))
axes = np.array(axes).reshape(-1, 3)
for i, gene in enumerate(top_svgs):
    r, c = divmod(i, 3)
    sq.pl.spatial_scatter(
        adata, color=gene, ax=axes[r, c], size=1.8,
        title=f"{gene}   I={moran_df.loc[gene,'I']:.3f}  p={moran_df.loc[gene,'pval_norm']:.1e}"
    )
for j in range(len(top_svgs), n_rows * 3):
    axes[j // 3, j % 3].axis("off")
plt.suptitle("Top Spatially Variable Genes — Moran's I (magma)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(output_dir / "08_top_svgs_spatial.png", dpi=150, bbox_inches="tight")
plt.close()

In [38]:
sq.gr.nhood_enrichment(adata, cluster_key="leiden", seed=42)

zscore_mat     = adata.uns["leiden_nhood_enrichment"]["zscore"]
cluster_labels = sorted(adata.obs["leiden"].unique(), key=int)
nhood_df       = pd.DataFrame(zscore_mat, index=cluster_labels, columns=cluster_labels)
nhood_df.to_csv(output_dir / "nhood_enrichment_zscore.csv")

  0%|          | 0/1000 [00:00<?, ?/s]

In [39]:
print(f"Z-score matrix shape: {nhood_df.shape}")

Z-score matrix shape: (11, 11)


In [40]:
nhood_df

,0,1,2,3,4,5,6,7,8,9,10
0,109.124100,-36.516702,-28.069589,-28.891184,-29.511038,-29.853795,-1.605565,-27.142024,-23.560574,-7.332984,-10.466383
1,-36.516702,111.015785,-26.855056,-5.832573,-25.537486,-17.802506,-22.709297,-12.759011,-15.170591,-5.163929,-8.698859
2,-28.069589,-26.855056,116.394579,-24.683276,-23.346198,-22.297722,-13.389254,-19.975814,-1.492530,-6.347447,-8.338391
3,-28.891184,-5.832573,-24.683276,72.913844,-3.305651,-11.470958,2.577466,-6.986913,-2.453110,-9.924645,12.950803
4,-29.511038,-25.537486,-23.346198,-3.305651,102.812747,-18.031098,-17.987945,-15.787019,-16.142803,-4.939891,-6.649932
5,-29.853795,-17.802506,-22.297722,-11.470958,-18.031098,122.879053,-19.927783,-2.575404,-16.025336,0.607091,-7.170396
6,-1.605565,-22.709297,-13.389254,2.577466,-17.987945,-19.927783,92.370198,-16.943650,-9.485040,-7.728479,-6.964376
7,-27.142024,-12.759011,-19.975814,-6.986913,-15.787019,-2.575404,-16.943650,97.768087,-14.409201,0.285429,-5.843993
8,-23.560574,-15.170591,-1.492530,-2.453110,-16.142803,-16.025336,-9.485040,-14.409201,106.552961,-2.103741,12.401014
9,-7.332984,-5.163929,-6.347447,-9.924645,-4.939891,0.607091,-7.728479,0.285429,-2.103741,26.737891,-3.798832


In [41]:
mask_upper = np.triu(np.ones(nhood_df.shape, dtype=bool), k=1)
for i in cluster_labels:
    for j in cluster_labels:
        if mask_upper[int(i), int(j)] and abs(nhood_df.loc[i, j]) > 5:
            print(f"    Cluster {i} <-> {j} : z = {nhood_df.loc[i, j]:.2f}")

fig, ax = plt.subplots(figsize=(10, 9))
sq.pl.nhood_enrichment(adata, cluster_key="leiden", ax=ax, show=False,
                        title="Neighborhood Enrichment Z-Score")
plt.tight_layout()
plt.savefig(output_dir / "09_nhood_enrichment.png", dpi=150, bbox_inches="tight")
plt.close()

    Cluster 0 <-> 1 : z = -36.52
    Cluster 0 <-> 2 : z = -28.07
    Cluster 0 <-> 3 : z = -28.89
    Cluster 0 <-> 4 : z = -29.51
    Cluster 0 <-> 5 : z = -29.85
    Cluster 0 <-> 7 : z = -27.14
    Cluster 0 <-> 8 : z = -23.56
    Cluster 0 <-> 9 : z = -7.33
    Cluster 0 <-> 10 : z = -10.47
    Cluster 1 <-> 2 : z = -26.86
    Cluster 1 <-> 3 : z = -5.83
    Cluster 1 <-> 4 : z = -25.54
    Cluster 1 <-> 5 : z = -17.80
    Cluster 1 <-> 6 : z = -22.71
    Cluster 1 <-> 7 : z = -12.76
    Cluster 1 <-> 8 : z = -15.17
    Cluster 1 <-> 9 : z = -5.16
    Cluster 1 <-> 10 : z = -8.70
    Cluster 2 <-> 3 : z = -24.68
    Cluster 2 <-> 4 : z = -23.35
    Cluster 2 <-> 5 : z = -22.30
    Cluster 2 <-> 6 : z = -13.39
    Cluster 2 <-> 7 : z = -19.98
    Cluster 2 <-> 9 : z = -6.35
    Cluster 2 <-> 10 : z = -8.34
    Cluster 3 <-> 5 : z = -11.47
    Cluster 3 <-> 7 : z = -6.99
    Cluster 3 <-> 9 : z = -9.92
    Cluster 3 <-> 10 : z = 12.95
    Cluster 4 <-> 5 : z = -18.03
    Cluster 4 <

In [42]:
sq.gr.co_occurrence(adata, cluster_key="leiden")
co_occ_data = adata.uns["leiden_co_occurrence"]

In [43]:
co_occ_data['occ'].shape

(11, 11, 49)

In [44]:
co_occ_data['interval'].shape[0]

50

In [45]:
plot_clusters = sorted(adata.obs["leiden"].unique(), key=int)[:4]
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for i, cl in enumerate(plot_clusters):
    r, c = divmod(i, 2)
    plt.sca(axes[r, c])

    sq.pl.co_occurrence(adata, cluster_key="leiden", clusters=cl)

    axes[r, c].set_title(f"Co-occurrence - Cluster {cl}", fontsize=12, fontweight="bold")

plt.suptitle("Spatial Co-occurrence Scores (vs. all clusters)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(output_dir / "10_co_occurrence.png", dpi=150, bbox_inches="tight")
plt.close()

In [46]:
sq.gr.ripley(adata, cluster_key="leiden", mode="L")
sq.gr.ripley(adata, cluster_key="leiden", mode="F")

In [47]:
adata.uns['leiden_ripley_L']

{'L_stat':             bins leiden       stats
 0       0.000000      0    0.000000
 1      98.140375      0    0.000000
 2     196.280749      0   74.977259
 3     294.421124      0  125.062978
 4     392.561498      0  157.426377
 ..           ...    ...         ...
 545  4416.316853      9   89.384067
 546  4514.457228      9   91.402910
 547  4612.597603      9   93.473399
 548  4710.737977      9   95.568895
 549  4808.878352      9   97.185299
 
 [550 rows x 3 columns],
 'sims_stat':              bins simulations        stats
 0        0.000000           0     0.000000
 1       98.140375           0    37.145716
 2      196.280749           0    74.291432
 3      294.421124           0   110.756059
 4      392.561498           0   147.425038
 ...           ...         ...          ...
 4995  4416.316853          99  1223.242413
 4996  4514.457228          99  1238.908473
 4997  4612.597603          99  1254.440988
 4998  4710.737977          99  1269.292680
 4999  4808.878352    

In [48]:
adata.uns['leiden_ripley_F']

{'F_stat':             bins leiden     stats
 0       0.000000      0  0.000000
 1      98.140375      0  0.152231
 2     196.280749      0  0.229921
 3     294.421124      0  0.266142
 4     392.561498      0  0.292913
 ..           ...    ...       ...
 545  4416.316853      9  1.000000
 546  4514.457228      9  1.000000
 547  4612.597603      9  1.000000
 548  4710.737977      9  1.000000
 549  4808.878352      9  1.000000
 
 [550 rows x 3 columns],
 'sims_stat':              bins simulations  stats
 0        0.000000           0  0.000
 1       98.140375           0  0.434
 2      196.280749           0  0.925
 3      294.421124           0  0.998
 4      392.561498           0  1.000
 ...           ...         ...    ...
 4995  4416.316853          99  1.000
 4996  4514.457228          99  1.000
 4997  4612.597603          99  1.000
 4998  4710.737977          99  1.000
 4999  4808.878352          99  1.000
 
 [5000 rows x 3 columns],
 'bins': array([   0.        ,   98.14037452, 

In [49]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
sq.pl.ripley(adata, cluster_key="leiden", mode="L", ax=axes[0])
axes[0].set_title("Ripley's L — Spatial Clustering", fontsize=12, fontweight="bold")

sq.pl.ripley(adata, cluster_key="leiden", mode="F", ax=axes[1])
axes[1].set_title("Ripley's F — Empty-Space Function", fontsize=12, fontweight="bold")

plt.suptitle("Ripley's Statistics per Leiden Cluster", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(output_dir / "11_ripley_statistics.png", dpi=150, bbox_inches="tight")
plt.close()

In [50]:
sq.gr.interaction_matrix(adata, cluster_key="leiden", normalized=True)
int_mat = adata.uns["leiden_interactions"]

In [51]:
int_mat.shape

(11, 11)

In [52]:
fig, ax = plt.subplots(figsize=(10, 9))
sq.pl.interaction_matrix(adata, cluster_key="leiden", ax=ax, show=False,
                           title="Normalised Cell–Cell Interaction Matrix")
plt.tight_layout()
plt.savefig(output_dir / "12_interaction_matrix.png", dpi=150, bbox_inches="tight")
plt.close()

In [53]:
sq.gr.centrality_scores(adata, cluster_key="leiden")
centrality_df = adata.uns["leiden_centrality_scores"]
print(f"Centrality scores (all clusters):\n{centrality_df.to_string()}")
centrality_df.to_csv(output_dir / "centrality_scores.csv")

sq.pl.centrality_scores(adata, cluster_key="leiden", figsize=(16, 5))
plt.suptitle("Centrality Scores per Leiden Cluster", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(output_dir / "13_centrality_scores.png", dpi=150, bbox_inches="tight")
plt.close()

Centrality scores (all clusters):
    degree_centrality  average_clustering  closeness_centrality
0            0.101540            0.560752              0.112034
1            0.109620            0.554857              0.156348
2            0.068981            0.561118              0.126129
3            0.271231            0.537348              0.329667
4            0.075692            0.559101              0.129553
5            0.074518            0.529198              0.140985
6            0.130841            0.530142              0.145389
7            0.078250            0.562668              0.164966
8            0.088990            0.533290              0.137179
9            0.186018            0.620909              0.359215
10           0.029921            0.535666              0.128186


In [54]:
sq.im.calculate_image_features(
    adata, img,
    features="summary",
    key_added="img_summary",
    n_jobs=1,
    show_progress_bar=True,
)

  0%|          | 0/2572 [00:00<?, ?/s]

In [55]:
adata.obsm['img_summary'].shape

(2572, 15)

In [56]:
sq.im.calculate_image_features(
    adata, img,
    features="histogram",
    key_added="img_histogram",
    n_jobs=1,
    show_progress_bar=True,
)

  0%|          | 0/2572 [00:00<?, ?/s]

In [57]:
adata.obsm['img_histogram'].shape

(2572, 30)

In [58]:
sq.im.calculate_image_features(
    adata, img,
    features="texture",
    key_added="img_texture",
    features_kwargs={"texture": {"distances": [1, 2],
                                  "angles":    [0, np.pi/4, np.pi/2, 3*np.pi/4]}},
    n_jobs=1,
    show_progress_bar=True,
)

  0%|          | 0/2572 [00:00<?, ?/s]

In [59]:
adata.obsm['img_texture'].shape

(2572, 120)

In [60]:
img_feat_parts = [adata.obsm[k] for k in ["img_summary", "img_histogram", "img_texture"]
                  if k in adata.obsm]
img_combined   = pd.concat(img_feat_parts, axis=1)
img_combined.index = adata.obs_names
adata.obsm["img_combined"] = img_combined

In [61]:
scaler_img  = StandardScaler()
img_scaled  = scaler_img.fit_transform(img_combined.values)
pca_img     = SklearnPCA(n_components=min(20, img_combined.shape[1]), random_state=42)
img_pca_emb = pca_img.fit_transform(img_scaled)
adata.obsm["X_img_pca"] = img_pca_emb
cumvar_img  = np.cumsum(pca_img.explained_variance_ratio_) * 100

In [62]:
cumvar_img[:5].round(1)

array([57.4, 72. , 80.3, 85.4, 87.5])

In [63]:
img_feat_display = img_combined.columns[:6].tolist()
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for i, feat in enumerate(img_feat_display):
    r, c = divmod(i, 3)
    col_tmp = f"_imgfeat_{i}"
    adata.obs[col_tmp] = img_combined[feat].values
    sq.pl.spatial_scatter(adata, color=col_tmp, ax=axes[r, c], size=1.8,
                           title=feat.replace("_", " "))
    del adata.obs[col_tmp]
plt.suptitle("H&E Image Features — Spatial Distribution",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(output_dir / "14_image_features_spatial.png", dpi=150, bbox_inches="tight")
plt.close()

In [64]:
corr_img = pd.DataFrame(
    np.corrcoef(img_scaled[:, :20].T),
    index=img_combined.columns[:20],
    columns=img_combined.columns[:20],
)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_img, ax=ax, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, cbar_kws={"shrink": 0.7},
            xticklabels=True, yticklabels=True)
ax.set_title("Image Feature Correlation Matrix (top 20 features)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(output_dir / "15_image_feature_correlation.png", dpi=150, bbox_inches="tight")
plt.close()

In [65]:
img_pc1 = img_pca_emb[:, 0]
img_pc2 = img_pca_emb[:, 1]
adata.obs["img_pc1"] = img_pc1
adata.obs["img_pc2"] = img_pc2

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
sq.pl.spatial_scatter(adata, color="img_pc1", ax=axes[0], size=1.8,
                       title=f"Image PC1 ({pca_img.explained_variance_ratio_[0]*100:.1f}% var)")
sq.pl.spatial_scatter(adata, color="img_pc2", ax=axes[1], size=1.8,
                       title=f"Image PC2 ({pca_img.explained_variance_ratio_[1]*100:.1f}% var)")
plt.suptitle("Top Image PCs — Spatial Distribution",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(output_dir / "16_image_pca_spatial.png", dpi=150, bbox_inches="tight")
plt.close()

In [66]:
known_markers = {
    "Oligodendrocytes": ["Mbp",    "Plp1", "Mog",    "Cnp",    "Mag"   ],
    "Astrocytes":       ["Gfap",   "Aqp4", "Aldh1l1","S100b",  "Vim"   ],
    "Excit_neurons":    ["Camk2a", "Gria1","Nrn1",   "Satb2",  "Cux1"  ],
    "Inhib_neurons":    ["Gad1",   "Gad2", "Pvalb",  "Sst",    "Vip"   ],
    "Microglia":        ["Cx3cr1", "P2ry12","Tmem119","Csf1r", "Hexb"  ],
    "Endothelial":      ["Cldn5",  "Pecam1","Tie1",  "Vwf",   "Esam"   ],
}


In [67]:
avail_markers = {
    ct: [g for g in genes if g in adata.var_names]
    for ct, genes in known_markers.items()
}

In [68]:
avail_markers = {ct: genes for ct, genes in avail_markers.items() if genes}

In [69]:
for ct, genes in avail_markers.items():
    print(f"{ct:<18}: {', '.join(genes)}")

Oligodendrocytes  : Mbp, Plp1, Mog, Cnp, Mag
Astrocytes        : Gfap, Aqp4, Aldh1l1, S100b, Vim
Excit_neurons     : Camk2a, Gria1, Nrn1, Satb2, Cux1
Inhib_neurons     : Gad1, Gad2, Pvalb, Sst, Vip
Microglia         : Cx3cr1, P2ry12, Tmem119, Csf1r, Hexb
Endothelial       : Cldn5, Pecam1, Tie1, Vwf, Esam


In [70]:
score_col_map = {}
for ct, genes in avail_markers.items():
    score_col = f"{ct}_score"
    sc.tl.score_genes(adata, gene_list=genes, score_name=score_col, use_raw=True)
    score_col_map[ct] = score_col

score_cols = list(score_col_map.values())
n_types    = len(score_cols)
n_cols_s   = 3
n_rows_s   = (n_types + n_cols_s - 1) // n_cols_s

fig, axes = plt.subplots(n_rows_s, n_cols_s, figsize=(18, 6 * n_rows_s))
axes = np.array(axes).reshape(-1, n_cols_s)
for i, (ct, sc_col) in enumerate(score_col_map.items()):
    r, c = divmod(i, n_cols_s)
    sq.pl.spatial_scatter(adata, color=sc_col, ax=axes[r, c],
                           size=1.8, title=ct.replace("_", " "))
for j in range(n_types, n_rows_s * n_cols_s):
    axes[j // n_cols_s, j % n_cols_s].axis("off")
plt.suptitle("Cell-Type Gene Module Scores — Spatial Distribution",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(output_dir / "17_celltype_scores_spatial.png", dpi=150, bbox_inches="tight")
plt.close()

In [71]:
fig, axes = plt.subplots(n_rows_s, n_cols_s, figsize=(18, 6 * n_rows_s))
axes = np.array(axes).reshape(-1, n_cols_s)
for i, (ct, sc_col) in enumerate(score_col_map.items()):
    r, c = divmod(i, n_cols_s)
    sc.pl.umap(adata, color=sc_col, ax=axes[r, c], show=False, color_map="RdBu_r",
               title=ct.replace("_", " "), frameon=False)
for j in range(n_types, n_rows_s * n_cols_s):
    axes[j // n_cols_s, j % n_cols_s].axis("off")
plt.suptitle("Cell-Type Gene Module Scores — UMAP", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(output_dir / "18_celltype_scores_umap.png", dpi=150, bbox_inches="tight")
plt.close()

In [72]:
try:
    sq.gr.ligrec(
        adata,
        n_perms=1000,
        cluster_key="leiden",
        use_raw=True,
        copy=False,
        key_added="leiden_ligrec",
        seed=42,
    )
    ligrec      = adata.uns["leiden_ligrec"]
    means_df    = ligrec["means"]
    pvals_df    = ligrec["pvalues"]
    n_tested    = means_df.shape[0]
    n_sig_lr    = (pvals_df < 0.05).values.sum()
    print(f"L-R pairs tested      : {n_tested:,}")
    print(f"Significant (p<0.05)  : {n_sig_lr:,}")

    means_df.to_csv(output_dir / "ligrec_means.csv")
    pvals_df.to_csv(output_dir / "ligrec_pvalues.csv")

    source_cls = sorted(adata.obs["leiden"].unique(), key=int)[:3]
    target_cls = sorted(adata.obs["leiden"].unique(), key=int)[:3]

    sq.pl.ligrec(
        adata,
        cluster_key="leiden",
        source_groups=source_cls,
        target_groups=target_cls,
        pvalue_threshold=0.05,
        remove_empty_interactions=True,
        show=False,
    )
    plt.suptitle("Ligand-Receptor Interactions (clusters 0–2 × 0–2, p<0.05)",
                 fontsize=12, fontweight="bold", y=1.01)
    plt.savefig(output_dir / "19_ligrec_interactions.png", dpi=150, bbox_inches="tight")
    plt.close()

except Exception as e:
    print(f"error: {e}")

0.00B [00:00, ?B/s]

0.00B [00:00, ?B/s]

0.00B [00:00, ?B/s]

  0%|          | 0/1000 [00:00<?, ?permutation/s]

L-R pairs tested      : 5,720
Significant (p<0.05)  : 172,715


In [73]:
cluster_summary = pd.DataFrame({
    "n_spots":         adata.obs["leiden"].value_counts().sort_index(),
    "mean_total_cnt":  adata.obs.groupby("leiden")["total_counts"].mean().round(1),
    "mean_n_genes":    adata.obs.groupby("leiden")["n_genes_by_counts"].mean().round(1),
    "mean_pct_mt":     adata.obs.groupby("leiden")["pct_counts_mt"].mean().round(3),
    "img_pc1_mean":    adata.obs.groupby("leiden")["img_pc1"].mean().round(4),
    "img_pc2_mean":    adata.obs.groupby("leiden")["img_pc2"].mean().round(4),
})
for ct, sc_col in score_col_map.items():
    cluster_summary[f"score_{ct}"] = (
        adata.obs.groupby("leiden")[sc_col].mean().round(4)
    )

cluster_summary.to_csv(output_dir / "cluster_summary.csv")

In [74]:
print(cluster_summary.to_string())

        n_spots  mean_total_cnt  mean_n_genes  mean_pct_mt  img_pc1_mean  img_pc2_mean  score_Oligodendrocytes  score_Astrocytes  score_Excit_neurons  score_Inhib_neurons  score_Microglia  score_Endothelial
leiden                                                                                                                                                                                                        
0           494     7051.899902        6078.5        0.917       -1.1505        0.9479                 -1.0135           -0.3475               0.0392               0.3066          -0.2556            -0.1339
1           337     6735.100098        6173.3        0.932        5.4110       -2.3308                  0.0558            0.1969              -0.8760               0.1570          -0.2572            -0.0784
2           296     6832.899902        5970.2        0.957       -9.8385        3.4652                 -1.0348           -0.1546              -0.1727               0.1801  

In [75]:
sig_svgs_out = moran_df[moran_df["pval_norm"] < 0.05].copy()
sig_svgs_out.to_csv(output_dir / "significant_SVGs_moran.csv")
sig_svgs_gc  = geary_df[geary_df["pval_norm"] < 0.05].copy()
sig_svgs_gc.to_csv(output_dir / "significant_SVGs_geary.csv")